In [16]:
import pandas as pd
import torch
from lightgbm import LGBMRegressor
from utils.loss_functions import r2_score
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import numpy as np
from pytoda.smiles import SMILESTokenizer

In [2]:
df_smiles_origin = pd.read_csv('../../data/DrugName_AND_Canonical_SMILES.csv')
df_smiles_origin.head()

,CID,SMILES,DRUG_NAME
0,3385,C1=C(C(=O)NC(=O)N1)F,5-Fluorouracil
1,9444,C1=NC(=NC(=O)N1C2C(C(C(O2)CO)O)O)N,5-azacytidine
2,76285486,COC1=C(C=C2C(=C1)C3(CCC3)C(=N2)N)OCCCN4CCCC4,A-366
3,11228183,CN(C)CCC(CSC1=CC=CC=C1)NC2=C(C=C(C=C2)S(=O)(=O...,ABT737
4,56645356,CC1=CC=CC=C1C(C(=O)NC2CCCCC2)N(C3=CC(=CC=C3)F)...,AGI-5198


In [3]:
smiles_language_filepath = '../../data/smiles_language/tokenizer_customized'
# Load SMILES language
smiles_language = SMILESTokenizer.from_pretrained(smiles_language_filepath)
smiles_language.set_encoding_transforms(
    add_start_and_stop=True,
    padding=True,
    padding_length=256,
    # padding_length=params.get("smiles_padding_length", None),
)
smiles_language.set_smiles_transforms(
    augment=False,
    canonical=False,
    kekulize=False,
    all_bonds_explicit=False,
    all_hs_explicit=False,
    remove_bonddir=False,
    remove_chirality=False,
    selfies=False,
    sanitize=False,
)
smiles_language.add_dataset(df_smiles_origin['SMILES'])

In [4]:
smiles = df_smiles_origin['SMILES'].values

In [5]:
# Convert all SMILES into tokens.
smiles_num_array = []
for smile in smiles:
    single_drug = smiles_language.smiles_to_token_indexes(smile)
    single_drug_num_array = np.array(single_drug)
    smiles_num_array.append(single_drug_num_array)

df_smiles = pd.DataFrame(smiles_num_array)

In [6]:
df_smiles.insert(0, 'drug', df_smiles_origin['DRUG_NAME'])
df_smiles.head()

,drug,0,1,2,3,4,5,6,7,8,...,246,247,248,249,250,251,252,253,254,255
0,5-Fluorouracil,0,0,0,0,0,0,0,0,0,...,38,4,37,35,5,36,6,5,43,3
1,5-azacytidine,0,0,0,0,0,0,0,0,0,...,5,38,35,5,35,5,35,5,36,3
2,A-366,0,0,0,0,0,0,0,0,0,...,38,38,36,9,38,38,38,38,9,3
3,ABT737,0,0,0,0,0,0,0,0,0,...,5,40,5,50,4,37,35,5,49,3
4,AGI-5198,0,0,0,0,0,0,0,0,0,...,9,38,37,38,36,37,38,9,38,3


In [7]:
df_drug_sensitivity = pd.read_csv('../../data/drug_sensitivity_All_CellLineDrugPairs.csv',index_col=0)
print(df_drug_sensitivity.shape)
df_drug_sensitivity.head()

(141222, 2)


,cell_line,IC50
drug,,
5-Fluorouracil,HL60,2.558926
5-azacytidine,HL60,0.917132
A-366,HL60,4.836160
ABT737,HL60,-2.817798
AGI-5198,HL60,3.644734


In [8]:
# GEP
df_GEP = pd.read_csv('../../data/GEP_Wilcoxon_Test_Analysis_Log10_P_value_C2_KEGG_MEDICUS.csv')
df_GEP.head()
# CNV
# df_CNV = pd.read_csv('../data/CNV_Cardinality_analysis_of_variance_Latest_MEDICUS.csv')
# MUT
# df_MUT = pd.read_csv('../data/MUT_cardinality_analysis_of_variance_Only_MEDICUS.csv')

,cell_line,KEGG_MEDICUS_ENV_FACTOR_ARSENIC_TO_ELECTRON_TRANSFER_IN_COMPLEX_IV,KEGG_MEDICUS_ENV_FACTOR_BENZO_A_PYRENRE_TO_CYP_MEDIATED_METABOLISM,KEGG_MEDICUS_ENV_FACTOR_BPA_TO_RAS_ERK_SIGNALING_PATHWAY,KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,KEGG_MEDICUS_ENV_FACTOR_E2_TO_NUCLEAR_INITIATED_ESTROGEN_SIGNALING_PATHWAY,KEGG_MEDICUS_ENV_FACTOR_E2_TO_RAS_ERK_SIGNALING_PATHWAY,KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT,KEGG_MEDICUS_ENV_FACTOR_METALS_TO_JNK_SIGNALING_PATHWAY,KEGG_MEDICUS_ENV_FACTOR_METALS_TO_KEAP1_NRF2_SIGNALIG_PATHWAY,...,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PERK_ATF4_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_VGCC_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_TEL_AML1_FUSION_TO_TRANSCRIPTIONAL_REPRESSION,KEGG_MEDICUS_VARIANT_TGFA_OVEREXPRESSION_TO_PI3K_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_TMPRSS2_ERG_FUSION_TO_TRANSCRIPTIONAL_ACTIVATION,KEGG_MEDICUS_VARIANT_TRK_FUSION_KINASE_TO_RAS_ERK_SIGNALING_PATHWAY
0,HL60,4.214134,0.440202,1.470915,0.205382,1.450508,1.482270,0.343427,0.748859,0.372432,...,17.685880,1.689896,2.603777,3.056268,1.462969,1.259628,1.490945,0.952237,0.182561,1.939787
1,HEL,3.646497,0.135204,1.566388,0.925705,1.111581,0.883885,0.949699,0.261281,0.576484,...,15.719172,1.853080,1.901783,1.311271,0.849546,0.863047,0.123552,0.346157,1.325299,2.701512
2,MONOMAC6,4.472972,0.014476,1.661451,0.019613,1.938533,1.503992,0.675432,0.944259,0.700977,...,17.283634,1.538003,3.062976,2.903670,1.140496,0.942227,0.708988,0.904325,0.409584,2.088677
3,LS513,4.352599,0.462321,1.699385,0.407889,0.665094,2.222235,0.398729,1.351826,1.652013,...,16.863173,1.824951,2.086926,0.256067,1.761924,0.436037,0.022385,0.619383,0.759916,1.872848
4,A101D,4.053421,0.062876,1.848867,0.094523,1.103970,1.862741,2.710302,0.520589,1.123209,...,19.239258,2.301589,3.182736,0.921441,1.438838,0.630094,0.104327,1.590609,0.198160,2.121071


In [9]:
def getTrainTestDataSet(path_train, path_test, fold):
    df_train = pd.read_csv(path_train.format(fold), index_col=0)
    df_train = pd.merge(df_train, df_GEP, on='cell_line')
    df_train = pd.merge(df_train, df_smiles , on='drug')
    df_train_X = df_train.iloc[:, 3:]
    df_train_y = df_train['IC50']
    
    df_test = pd.read_csv(path_test.format(fold), index_col=0)
    df_test = pd.merge(df_test, df_GEP, on='cell_line')
    df_test = pd.merge(df_test, df_smiles , on='drug')
    df_test_X = df_test.iloc[:, 3:]
    df_test_y = df_test['IC50']
    
    return df_train_X, df_test_X, df_train_y, df_test_y

In [18]:
n_folds = 10
path_test = '../../data/10_fold_data/mixed/MixedSet_test_Fold{}.csv'
path_train = '../../data/10_fold_data/mixed/MixedSet_train_Fold{}.csv'

print('start training......')
# 交叉验证
for i in range(n_folds):
    print('######## ', 'Fold[', i + 1, '] ########')
    X_train, X_test, y_train, y_test = getTrainTestDataSet(path_train, path_test, i)
    X_train.columns = range(X_train.shape[1])
    X_test.columns = range(X_test.shape[1])
    X_train = X_train.astype('float32').to_numpy()
    X_test = X_test.astype('float32').to_numpy()
    y_train = y_train.astype('float32').to_numpy()
    y_test = y_test.astype('float32').to_numpy()

    LGBM = LGBMRegressor()
    LGBM.fit(X_train, y_train)

    predictions = LGBM.predict(X_test)

    LGBM_mse = mean_squared_error(y_test, predictions)
    LGBM_rmse = np.sqrt(LGBM_mse)
    LGBM_r2 = r2_score(torch.Tensor(y_test), torch.Tensor(predictions)).item()
    LGBM_pearson = np.corrcoef(predictions, y_test)[0, 1]
    print('LGBM MSE:', LGBM_mse)
    print('LGBM RMSE:', LGBM_rmse)
    print('LGBM R2:', LGBM_r2)
    print('LGBM Pearson:', LGBM_pearson)

    df_rf = pd.DataFrame({'y_test': y_test, 'LGBM_pred': predictions})
    df_rf.to_csv('result/LGBM/LGBM_prediction_fold{}.csv'.format(i))

start training......
########  Fold[ 1 ] ########
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.208040 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 158790
[LightGBM] [Info] Number of data points in the train set: 126876, number of used features: 805
[LightGBM] [Info] Start training from score 2.809198
LGBM MSE: 1.3870696513826597
LGBM RMSE: 1.1777392119576642
LGBM R2: 0.8228785991668701
LGBM Pearson: 0.9079445288483341
########  Fold[ 2 ] ########
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.200825 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 158791
[LightGBM] [Info] Number of data points in the train set: 126904, number of used features: 805
[LightGBM] [Info] Start training from score 2.810994
LGBM MSE: 1.450593208202704
LGBM RMSE: 1.204405748991055
LGBM R2: 0.8156354427337646
LGBM Pearson: 0.90415015